# Monitoramento de Degradação de Rolamentos (NASA IMS Dataset)
<i>Bearing degradation monitoring system</i>

Este notebook orquestra o pipeline completo de manutenção preditiva, desde a extração de características 
estatísticas de sinais de vibração brutos até a predição do tempo de vida útil restante (RUL) 
e a implementação de uma lógica de alertas de saúde (Health Index).

**Workflow:**
1. **Ingestão:** Leitura de arquivos .txt (amostras de 1s a 20kHz).
2. **Feature Engineering:** Extração de métricas no domínio do tempo (RMS, Kurtosis, etc).
3. **Modelagem:** Comparação entre Baselines (Ridge, RF) e o modelo final (XGBoost Otimizado).
4. **Health Index:** Desenvolvimento de um indicador de saúde robusto e suavizado.

In [1]:
#%load_ext autoreload
#%autoreload 2

import os
import glob
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from scipy.stats import kurtosis, skew
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from datetime import datetime
from collections import Counter
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import seaborn as sns
from matplotlib.colors import ListedColormap


import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

from src.config.config import DATA_RAW
from src.loaders.csv_loader import CSVLoader

csvLoader = CSVLoader()

In [2]:
df = csvLoader.create_dataframe(DATA_RAW,"synthetic_beverage_sales_data.csv")

In [ ]:
df["Quantity"] = df["Quantity"].fillna(0)

In [ ]:
print(df.shape)


In [3]:
print (df.head())

  Order_ID Customer_ID Customer_Type             Product     Category  \
0     ORD1     CUS1496           B2B          Vio Wasser        Water   
1     ORD1     CUS1496           B2B               Evian        Water   
2     ORD1     CUS1496           B2B              Sprite  Soft Drinks   
3     ORD1     CUS1496           B2B  Rauch Multivitamin       Juices   
4     ORD1     CUS1496           B2B        Gerolsteiner        Water   

   Unit_Price  Quantity  Discount  Total_Price             Region  Order_Date  
0        1.66        53      0.10        79.18  Baden-Württemberg  2023-08-23  
1        1.56        90      0.10       126.36  Baden-Württemberg  2023-08-23  
2        1.17        73      0.05        81.14  Baden-Württemberg  2023-08-23  
3        3.22        59      0.10       170.98  Baden-Württemberg  2023-08-23  
4        0.87        35      0.10        27.40  Baden-Württemberg  2023-08-23  


In [ ]:
df["Customer_ID"].nunique()

In [4]:
dfx = df.copy()

dfx["Quantity"] = pd.to_numeric(dfx["Quantity"], errors="coerce").fillna(0)

# 1. Descobre as 15 regiões com maior quantidade total
top_15_regions = (
    dfx.groupby("Region")["Quantity"]
       .sum()
       .nlargest(15)
       .index
)

# 2. Filtra somente essas regiões
dfx = dfx[dfx["Region"].isin(top_15_regions)].copy()

# 3. Ordena para cálculo temporal
dfx = dfx.sort_values(["Product", "Region", "Order_Date"]).copy()

# 4. Calcula a média móvel por Produto + Região
dfx["sales_mean_period"] = (
    dfx.groupby(["Product", "Region"])["Quantity"]
       .transform(lambda s: s.shift(1).rolling(15, min_periods=3).mean())
)

print(dfx.head())

          Order_ID Customer_ID Customer_Type      Product Category  \
201824    ORD67170     CUS5272           B2B  Apollinaris    Water   
1634002  ORD544575     CUS3139           B2C  Apollinaris    Water   
1688107  ORD562480     CUS7034           B2C  Apollinaris    Water   
2030242  ORD676521     CUS7976           B2C  Apollinaris    Water   
2370387  ORD789632     CUS6655           B2B  Apollinaris    Water   

         Unit_Price  Quantity  Discount  Total_Price             Region  \
201824         0.45        24      0.05        10.26  Baden-Württemberg   
1634002        0.93        15      0.00        13.95  Baden-Württemberg   
1688107        0.93         7      0.00         6.51  Baden-Württemberg   
2030242        0.93        14      0.00        13.02  Baden-Württemberg   
2370387        1.13        99      0.15        95.09  Baden-Württemberg   

         Order_Date  sales_mean_period  
201824   2021-01-01                NaN  
1634002  2021-01-01                NaN  
16881

In [5]:
top_15_regions_df = (
    dfx.groupby("Region", as_index=False)["Quantity"]
      .sum()
      .sort_values("Quantity", ascending=False)
      .head(15)
)

print(top_15_regions_df)

                    Region  Quantity
4                  Hamburg  14327571
10                Saarland  13785088
5                   Hessen  13644010
9          Rheinland-Pfalz  13596775
6   Mecklenburg-Vorpommern  13244958
3                   Bremen  13186325
14               Thüringen  13068359
2                   Berlin  12869396
8      Nordrhein-Westfalen  12786649
1                   Bayern  12679489
11                 Sachsen  12646534
7            Niedersachsen  12545661
12          Sachsen-Anhalt  12539664
0        Baden-Württemberg  12524670
13      Schleswig-Holstein  12402059


In [6]:
#dfx = df.copy()

#dfx["Quantity"] = pd.to_numeric(dfx["Quantity"], errors="coerce").fillna(0)

top_15_region_product = (
    dfx.groupby(["Region", "Product"], as_index=False)["Quantity"]
       .sum()
       .sort_values("Quantity", ascending=False)
       .head(15)
)

dfx = dfx.merge(
    top_15_region_product[["Region", "Product"]],
    on=["Region", "Product"],
    how="inner"
)

dfx = dfx.sort_values(["Quantity","Product", "Region", "Order_Date"], ascending=[False, True, True, True]).copy()


dfx["sales_mean_period"] = (
    dfx.groupby(["Product", "Region"])["Quantity"]
       .transform(lambda s: s.shift(1).rolling(15, min_periods=3).mean())
)

print(dfx.head())

        Order_ID Customer_ID Customer_Type Product             Category  \
670   ORD1532747      CUS645           B2B  Beck's  Alcoholic Beverages   
751    ORD941979     CUS4509           B2B  Beck's  Alcoholic Beverages   
871    ORD881079     CUS6088           B2B  Beck's  Alcoholic Beverages   
1029   ORD131198     CUS8754           B2B  Beck's  Alcoholic Beverages   
1199  ORD2643564     CUS5055           B2B  Beck's  Alcoholic Beverages   

      Unit_Price  Quantity  Discount  Total_Price  Region  Order_Date  \
670         1.38       100       0.1        124.2  Bremen  2021-01-23   
751         2.09       100       0.1        188.1  Bremen  2021-01-26   
871         0.93       100       0.1         83.7  Bremen  2021-01-30   
1029        2.01       100       0.1        180.9  Bremen  2021-02-05   
1199        2.01       100       0.1        180.9  Bremen  2021-02-10   

      sales_mean_period  
670                 NaN  
751                 NaN  
871                 NaN  
1029  

In [7]:
dfx = df.copy()

# Ajustes básicos
dfx["Quantity"] = pd.to_numeric(dfx["Quantity"], errors="coerce").fillna(0)
dfx["Order_Date"] = pd.to_datetime(dfx["Order_Date"])

# Define os últimos 15 dias com base na data mais recente do dataset
data_final = dfx["Order_Date"].max()
data_inicial = data_final - pd.Timedelta(days=15)

# Filtra o período
df_15d = dfx[(dfx["Order_Date"] >= data_inicial) & (dfx["Order_Date"] <= data_final)].copy()

# Agrupa por Região + Produto e soma a quantidade
top_vendas_15d = (
    df_15d.groupby(["Region", "Product"], as_index=False)["Quantity"]
          .sum()
          .sort_values("Quantity", ascending=False)
)

print(top_vendas_15d.head(20))

                     Region              Product  Quantity
105                  Berlin           Fritz-Kola     12160
191                  Bremen               Beck's     11430
297                  Hessen       Hohes C Orange      9954
132                  Berlin               Sprite      9906
246                 Hamburg           Fritz-Kola      9904
194                  Bremen            Coca-Cola      9801
546                Saarland   Rauch Multivitamin      8912
248                 Hamburg        Granini Apple      8568
491         Rheinland-Pfalz          Mango Juice      8430
358  Mecklenburg-Vorpommern   Rauch Multivitamin      8321
391           Niedersachsen       Hohes C Orange      8246
269                 Hamburg       San Pellegrino      8207
510         Rheinland-Pfalz         Tomato Juice      8202
107                  Berlin        Granini Apple      8060
544                Saarland  Passion Fruit Juice      7959
497         Rheinland-Pfalz  Passion Fruit Juice      78

In [8]:
dfx = df.copy()

dfx["Quantity"] = pd.to_numeric(dfx["Quantity"], errors="coerce").fillna(0)
dfx["Order_Date"] = pd.to_datetime(dfx["Order_Date"])

data_inicial = pd.to_datetime("2021-01-01")
data_final   = pd.to_datetime("2021-01-31")

df_15d = dfx[(dfx["Order_Date"] >= data_inicial) & (dfx["Order_Date"] <= data_final)].copy()

top_vendas_15d = (
    df_15d.groupby(["Region", "Product"], as_index=False)["Quantity"]
          .sum()
          .sort_values("Quantity", ascending=False)
)

print(top_vendas_15d.head(15))

                 Region              Product  Quantity
105              Berlin           Fritz-Kola     20985
191              Bremen               Beck's     19970
194              Bremen            Coca-Cola     19282
246             Hamburg           Fritz-Kola     18474
297              Hessen       Hohes C Orange     18188
132              Berlin               Sprite     18145
720           Thüringen       Hohes C Orange     16499
262             Hamburg  Passion Fruit Juice     16207
269             Hamburg       San Pellegrino     16204
391       Niedersachsen       Hohes C Orange     15958
491     Rheinland-Pfalz          Mango Juice     15665
264             Hamburg   Rauch Multivitamin     15178
485     Rheinland-Pfalz       Hohes C Orange     15019
685  Schleswig-Holstein  Passion Fruit Juice     14951
242             Hamburg      Cranberry Juice     14897


In [ ]:
import pandas as pd
import plotly.graph_objects as go

# =========================
# 1. Cópia e tratamento
# =========================
dfx = df.copy()

dfx["Quantity"] = pd.to_numeric(dfx["Quantity"], errors="coerce").fillna(0)
dfx["Order_Date"] = pd.to_datetime(dfx["Order_Date"], errors="coerce")

# =========================
# 2. Definir período de 15 dias
#    (últimos 15 dias com base na maior data do dataset)
# =========================
data_final = dfx["Order_Date"].max()
data_inicial = data_final - pd.Timedelta(days=15)

df_15d = dfx[
    (dfx["Order_Date"] >= data_inicial) &
    (dfx["Order_Date"] <= data_final)
].copy()

# =========================
# 3. Top 10 produtos por quantidade
# =========================
top_products = (
    df_15d.groupby("Category", as_index=False)["Quantity"]
    .sum()
    .sort_values("Quantity", ascending=False)
    .head(10)
)

top_products_list = top_products["Category"].tolist()

# =========================
# 4. Top 15 regiões por quantidade
# =========================
top_regions = (
    df_15d.groupby("Region", as_index=False)["Quantity"]
    .sum()
    .sort_values("Quantity", ascending=False)
    .head(15)
)

top_regions_list = top_regions["Region"].tolist()

# =========================
# 5. Manter apenas Produto x Região dentro dos tops
# =========================
df_sankey = df_15d[
    df_15d["Category"].isin(top_products_list) &
    df_15d["Region"].isin(top_regions_list)
].copy()

# =========================
# 6. Agregar fluxo Produto -> Região
# =========================
flows = (
    df_sankey.groupby(["Category", "Region"], as_index=False)["Quantity"]
    .sum()
    .sort_values("Quantity", ascending=False)
)

# =========================
# 7. Montar nós
# =========================
products_nodes = top_products_list
regions_nodes = top_regions_list
all_nodes = products_nodes + regions_nodes

node_indices = {name: i for i, name in enumerate(all_nodes)}

flows["source"] = flows["Category"].map(node_indices)
flows["target"] = flows["Region"].map(node_indices)

# =========================
# 8. Cores pastéis claras
# =========================
product_colors = [
    "#DCEEF2", "#FBE4D8", "#E7F4E8", "#F6E8F5", "#FFF4CC",
    "#E3F0FF", "#FDECEF", "#EAF7F2", "#F3ECFF", "#FFF0E1"
]

region_colors = [
    "#CFE8F3", "#F9DCC4", "#D9F0D8", "#EADCF8", "#FAEDCB",
    "#D6EAF8", "#F8D7E3", "#DFF3EA", "#E8E2FA", "#FCE8D5",
    "#D4EEF6", "#F7E1D3", "#DDF2DD", "#EFE3F7", "#FFF2D8"
]

node_colors = product_colors[:len(products_nodes)] + region_colors[:len(regions_nodes)]

# Para os links, usar cor bem suave/transparente
link_color = "rgba(180, 190, 200, 0.20)"
link_colors = [link_color] * len(flows)

# =========================
# 9. Sankey
# =========================
fig = go.Figure(data=[go.Sankey(
    arrangement="snap",
    node=dict(
        pad=18,
        thickness=18,
        line=dict(color="rgba(120,120,120,0.20)", width=0.6),
        label=all_nodes,
        color=node_colors,
        hovertemplate="%{label}<extra></extra>"
    ),
    link=dict(
        source=flows["source"],
        target=flows["target"],
        value=flows["Quantity"],
        color=link_colors,
        hovertemplate=(
            "Origem: %{source.label}<br>" +
            "Destino: %{target.label}<br>" +
            "Quantidade: %{value}<extra></extra>"
        )
    )
)])

fig.update_layout(
    title={
        "text": f"Sankey - Top 10 Produtos → Top 15 Regiões<br><sup>Período: {data_inicial.date()} até {data_final.date()}</sup>",
        "x": 0.5,
        "xanchor": "center"
    },
    font=dict(size=12, color="#4A5568"),
    paper_bgcolor="white",
    plot_bgcolor="white",
    width=1200,
    height=750,
    margin=dict(l=30, r=30, t=80, b=30)
)

fig.show()

In [ ]:
import plotly.graph_objects as go

def plot_sankey_top_products_regions(
    df,
    days=15,
    top_n_products=10,
    top_n_regions=15,
    limit=0
):
    dfx = df.copy()
    dfx["Quantity"] = pd.to_numeric(dfx["Quantity"], errors="coerce").fillna(0)
    dfx["Order_Date"] = pd.to_datetime(dfx["Order_Date"], errors="coerce")

    data_final = dfx["Order_Date"].max()
    data_inicial = data_final - pd.Timedelta(days=days)

    df_period = dfx[
        (dfx["Order_Date"] >= data_inicial) &
        (dfx["Order_Date"] <= data_final)
    ].copy()

    top_products = (
        df_period.groupby("Product", as_index=False)["Quantity"]
        .sum()
        .sort_values("Quantity", ascending=False)
        .head(top_n_products)
    )

    top_regions = (
        df_period.groupby("Region", as_index=False)["Quantity"]
        .sum()
        .sort_values("Quantity", ascending=False)
        .head(top_n_regions)
    )

    top_products_list = top_products["Product"].tolist()
    top_regions_list = top_regions["Region"].tolist()

    df_sankey = df_period[
        df_period["Product"].isin(top_products_list) &
        df_period["Region"].isin(top_regions_list)
    ].copy()

    flows = (
        df_sankey.groupby(["Product", "Region"], as_index=False)["Quantity"]
        .sum()
        .sort_values("Quantity", ascending=False)
    )

    # Aplica o limite
    if limit > 0:
        flows = flows[flows["Quantity"] >= limit].copy()

    if flows.empty:
        print(f"Nenhum fluxo encontrado com Quantity >= {limit}.")
        return

    # Mantém apenas nós realmente usados após o filtro
    products_nodes = flows["Product"].drop_duplicates().tolist()
    regions_nodes = flows["Region"].drop_duplicates().tolist()
    all_nodes = products_nodes + regions_nodes

    node_indices = {name: i for i, name in enumerate(all_nodes)}

    flows["source"] = flows["Product"].map(node_indices)
    flows["target"] = flows["Region"].map(node_indices)

    product_colors = [
        "#DCEEF2", "#FBE4D8", "#E7F4E8", "#F6E8F5", "#FFF4CC",
        "#E3F0FF", "#FDECEF", "#EAF7F2", "#F3ECFF", "#FFF0E1"
    ]
    region_colors = [
        "#CFE8F3", "#F9DCC4", "#D9F0D8", "#EADCF8", "#FAEDCB",
        "#D6EAF8", "#F8D7E3", "#DFF3EA", "#E8E2FA", "#FCE8D5",
        "#D4EEF6", "#F7E1D3", "#DDF2DD", "#EFE3F7", "#FFF2D8"
    ]

    node_colors = (
        product_colors[:len(products_nodes)] +
        region_colors[:len(regions_nodes)]
    )

    # Se faltar cor, completa com um tom neutro claro
    if len(node_colors) < len(all_nodes):
        node_colors += ["#E8EDF3"] * (len(all_nodes) - len(node_colors))

    link_colors = ["rgba(180, 190, 200, 0.20)"] * len(flows)

    fig = go.Figure(data=[go.Sankey(
        arrangement="snap",
        node=dict(
            pad=18,
            thickness=18,
            line=dict(color="rgba(120,120,120,0.20)", width=0.6),
            label=all_nodes,
            color=node_colors
        ),
        link=dict(
            source=flows["source"],
            target=flows["target"],
            value=flows["Quantity"],
            color=link_colors,
            hovertemplate=(
                "Origem: %{source.label}<br>" +
                "Destino: %{target.label}<br>" +
                "Quantidade: %{value}<extra></extra>"
            )
        )
    )])

    fig.update_layout(
        title=(
            f"Sankey - Top {top_n_products} Produtos → Top {top_n_regions} Regiões "
            f"({data_inicial.date()} a {data_final.date()}) | Limit = {limit}"
        ),
        font=dict(size=12, color="#4A5568"),
        paper_bgcolor="white",
        plot_bgcolor="white",
        width=1200,
        height=750
    )

    fig.show()

In [ ]:
plot_sankey_top_products_regions(df, days=15, top_n_products=10, top_n_regions=15, limit=5000)

In [9]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

# opcional para VS Code / ambiente local
pio.renderers.default = "browser"

def plot_sankey_top_products_regions(
    df,
    days=15,
    top_n_products=10,
    top_n_regions=15,
    limit=0,
    filter_mode="flow"   # "flow", "node", "both"
):
    dfx = df.copy()

    # Tratamento básico
    dfx["Quantity"] = pd.to_numeric(dfx["Quantity"], errors="coerce").fillna(0)
    dfx["Order_Date"] = pd.to_datetime(dfx["Order_Date"], errors="coerce")

    # Período
    data_final = dfx["Order_Date"].max()
    data_inicial = data_final - pd.Timedelta(days=days)

    df_period = dfx[
        (dfx["Order_Date"] >= data_inicial) &
        (dfx["Order_Date"] <= data_final)
    ].copy()

    # -----------------------------
    # TOP produtos e regiões
    # -----------------------------
    top_products = (
        df_period.groupby("Product", as_index=False)["Quantity"]
        .sum()
        .sort_values("Quantity", ascending=False)
    )

    top_regions = (
        df_period.groupby("Region", as_index=False)["Quantity"]
        .sum()
        .sort_values("Quantity", ascending=False)
    )

    # Filtro no nível dos nós
    if limit > 0 and filter_mode in ["node", "both"]:
        top_products = top_products[top_products["Quantity"] >= limit].copy()
        top_regions = top_regions[top_regions["Quantity"] >= limit].copy()

    top_products = top_products.head(top_n_products)
    top_regions = top_regions.head(top_n_regions)

    top_products_list = top_products["Product"].tolist()
    top_regions_list = top_regions["Region"].tolist()

    # Filtra dataset para os tops
    df_sankey = df_period[
        df_period["Product"].isin(top_products_list) &
        df_period["Region"].isin(top_regions_list)
    ].copy()

    # -----------------------------
    # Fluxos Produto -> Região
    # -----------------------------
    flows = (
        df_sankey.groupby(["Product", "Region"], as_index=False)["Quantity"]
        .sum()
        .sort_values("Quantity", ascending=False)
    )

    # Filtro no nível do fluxo
    if limit > 0 and filter_mode in ["flow", "both"]:
        flows = flows[flows["Quantity"] >= limit].copy()

    if flows.empty:
        print(f"Nenhum dado encontrado para limit={limit} e filter_mode='{filter_mode}'.")
        return

    # Mantém apenas nós realmente usados
    products_nodes = flows["Product"].drop_duplicates().tolist()
    regions_nodes = flows["Region"].drop_duplicates().tolist()
    all_nodes = products_nodes + regions_nodes

    node_indices = {name: i for i, name in enumerate(all_nodes)}

    flows["source"] = flows["Product"].map(node_indices)
    flows["target"] = flows["Region"].map(node_indices)

    # Cores pastéis claras
    product_colors = [
        "#DCEEF2", "#FBE4D8", "#E7F4E8", "#F6E8F5", "#FFF4CC",
        "#E3F0FF", "#FDECEF", "#EAF7F2", "#F3ECFF", "#FFF0E1"
    ]
    region_colors = [
        "#CFE8F3", "#F9DCC4", "#D9F0D8", "#EADCF8", "#FAEDCB",
        "#D6EAF8", "#F8D7E3", "#DFF3EA", "#E8E2FA", "#FCE8D5",
        "#D4EEF6", "#F7E1D3", "#DDF2DD", "#EFE3F7", "#FFF2D8"
    ]

    node_colors = (
        product_colors[:len(products_nodes)] +
        region_colors[:len(regions_nodes)]
    )

    if len(node_colors) < len(all_nodes):
        node_colors += ["#E8EDF3"] * (len(all_nodes) - len(node_colors))

    link_colors = ["rgba(180, 190, 200, 0.20)"] * len(flows)

    fig = go.Figure(data=[go.Sankey(
        arrangement="snap",
        node=dict(
            pad=18,
            thickness=18,
            line=dict(color="rgba(120,120,120,0.20)", width=0.6),
            label=all_nodes,
            color=node_colors
        ),
        link=dict(
            source=flows["source"],
            target=flows["target"],
            value=flows["Quantity"],
            color=link_colors,
            hovertemplate=(
                "Origem: %{source.label}<br>"
                "Destino: %{target.label}<br>"
                "Quantidade: %{value}<extra></extra>"
            )
        )
    )])

    fig.update_layout(
        title=(
            f"Sankey - Top {top_n_products} Produtos → Top {top_n_regions} Regiões<br>"
            f"<sup>Período: {data_inicial.date()} até {data_final.date()} | "
            f"limit={limit} | filter_mode='{filter_mode}'</sup>"
        ),
        font=dict(size=12, color="#4A5568"),
        paper_bgcolor="white",
        plot_bgcolor="white",
        width=1200,
        height=750,
        margin=dict(l=30, r=30, t=80, b=30)
    )

    fig.show()

In [11]:
plot_sankey_top_products_regions(
    df,
    days=15,
    top_n_products=10,
    top_n_regions=5,
    limit=40000,
    filter_mode="node"
)